# 📊 Weinstein Triple Confirmation Pattern — Backtester

## What does this notebook do?

This notebook runs Stan Weinstein's **Triple Confirmation Pattern** across **~500 NSE-listed stocks** and checks every signal that fired over the last 5+ years (2019–2024).

For every signal it finds, it also measures:
- How the stock performed **1 month later**
- How the stock performed **3 months later**
- How the stock performed **6 months later**
- How the stock performed **1 year later**

The results are saved as a **CSV file** you can download and open in Excel.

---

## How to run this notebook (3 simple steps)

1. **Read Section 2** (Configuration) and decide if you want a quick test (20 stocks, ~2 minutes) or full scan (500 stocks, ~15 minutes)
2. Click **Run → Run All Cells** from the menu at the top
3. When finished, go to the `results/` folder on the left panel, right-click the CSV file, and click **Download**

> **Note:** The very first time you open this Codespace, please wait 60–90 seconds for the setup to finish before clicking Run All Cells.

---

## Section 1 — Library Check

This cell confirms all required libraries are installed. If you see **"All libraries loaded successfully"**, you are good to go. If you see an error, please open a Terminal (top menu → Terminal → New Terminal) and type:
```
pip install -r requirements.txt
```
Then re-run this cell.

In [ ]:
import importlib, sys

required = ['yfinance', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'tqdm', 'openpyxl']
missing  = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"MISSING libraries: {missing}")
    print("Please open Terminal → run: pip install -r requirements.txt")
else:
    import pandas as pd
    import numpy  as np
    import yfinance as yf
    import matplotlib.pyplot as plt
    import seaborn as sns
    print("✅ All libraries loaded successfully!")
    print(f"   pandas {pd.__version__} | numpy {np.__version__} | yfinance {yf.__version__}")

---

## Section 2 — ⚙️ Configuration (Only section you need to change)

You can change the values below:

| Setting | What it does |
|---|---|
| `START_DATE` | Start of the backtest period |
| `END_DATE` | End of the backtest period |
| `QUICK_TEST` | **True** = scan only 20 stocks (~2 min). **False** = scan all ~500 stocks (~15 min) |
| `VOL_MULT` | Volume must be this many times higher than average (default: 2.0) |
| `RANGE_MULT` | Weekly range must be this many times the average true range (default: 2.0) |

In [ ]:
# ================================================================
# CONFIGURATION — Change values here if needed
# ================================================================

START_DATE  = "2019-01-01"   # Start of backtest  (format: YYYY-MM-DD)
END_DATE    = "2024-12-31"   # End of backtest    (format: YYYY-MM-DD)

QUICK_TEST  = True           # True = 20 stocks (~2 min) | False = all stocks (~15 min)

VOL_MULT    = 2.0            # Volume spike multiplier (Pine Script default: 2.0)
RANGE_MULT  = 2.0            # Range multiplier        (Pine Script default: 2.0)

# ================================================================
print(f"Backtest period : {START_DATE}  →  {END_DATE}")
print(f"Quick test mode : {QUICK_TEST}")
print(f"Volume filter   : {VOL_MULT}x average volume")
print(f"Range filter    : {RANGE_MULT}x average true range")

---

## Section 3 — Load Stock Universe

This loads the list of NSE stocks we will scan.

In [ ]:
import pandas as pd

universe = pd.read_csv("data/nifty500_symbols.csv")

# Remove any accidental duplicates from the CSV
universe = universe.drop_duplicates(subset="symbol").reset_index(drop=True)

all_symbols = universe["symbol"].tolist()

if QUICK_TEST:
    # Use first 20 stocks for a quick sanity check
    symbols = all_symbols[:20]
    print(f"⚡ QUICK TEST MODE: scanning {len(symbols)} stocks")
    print(f"   To run all stocks, set QUICK_TEST = False in Section 2")
else:
    symbols = all_symbols
    print(f"🔍 FULL SCAN MODE: scanning {len(symbols)} stocks")
    print(f"   Estimated time: 12–18 minutes (depends on internet speed)")

print(f"\nSample of stocks to be scanned:")
universe[universe["symbol"].isin(symbols[:10])].reset_index(drop=True)

---

## Section 4 — 🚀 Run the Backtest

This is the main step. It will:
1. Download weekly price data for all stocks from Yahoo Finance (free)
2. Calculate all indicators (30W SMA, ATR, Mansfield RS, etc.)
3. Find every week where **all 6 Triple Confirmation conditions** fired
4. Measure how each stock performed after each signal

A progress bar will show the download progress. **Please wait for it to complete.**

In [ ]:
from triple_confirmation import download_weekly_data, run_backtest

# Step 1: Download all data
print("Step 1 of 2: Downloading historical data from Yahoo Finance...")
print("(This may take a few minutes. A progress bar will appear below.)\n")

data = download_weekly_data(
    symbols  = symbols,
    start    = START_DATE,
    end      = END_DATE,
)

print("\nStep 2 of 2: Scanning for Triple Confirmation signals...\n")

results = run_backtest(
    data       = data,
    vol_mult   = VOL_MULT,
    range_mult = RANGE_MULT,
)

---

## Section 5 — 💾 Save Results

Results are saved to the `results/` folder as a CSV file.

**To download the file:** Right-click `results/signals_....csv` in the left panel and select **Download**.

In [ ]:
import os

os.makedirs("results", exist_ok=True)

if results.empty:
    print("No signals found in the selected period.")
    print("Try expanding the date range or reducing the filters.")
else:
    # Save CSV
    csv_path  = f"results/signals_{START_DATE[:4]}_{END_DATE[:4]}.csv"
    xlsx_path = f"results/signals_{START_DATE[:4]}_{END_DATE[:4]}.xlsx"

    results.to_csv(csv_path, index=False)
    results.to_excel(xlsx_path, index=False)

    print(f"✅ Results saved!")
    print(f"   CSV  → {csv_path}")
    print(f"   Excel → {xlsx_path}")
    print(f"\n📋 Showing first 10 signals:")

    # Format for display
    display_cols = ["symbol", "signal_date", "signal_price", "stage",
                    "mansfield_rs", "ret_1m", "ret_3m", "ret_6m", "ret_1y"]

    display_df = results[display_cols].head(10).copy()
    display_df.columns = ["Stock", "Signal Date", "Price (₹)", "Stage",
                          "Mansfield RS", "Return 1M %", "Return 3M %",
                          "Return 6M %", "Return 1Y %"]
    display_df

---

## Section 6 — 📊 Summary Statistics

How often did the pattern lead to profits? This section shows the win rate and average returns.

In [ ]:
from triple_confirmation import print_summary

if not results.empty:
    print_summary(results)

    print("\n📅 Signals by Year:")
    results["year"] = pd.to_datetime(results["signal_date"]).dt.year
    by_year = results.groupby("year").size().reset_index(name="signal_count")
    print(by_year.to_string(index=False))

    print("\n🏭 Top 15 Most-Signalled Stocks:")
    top_stocks = results["symbol"].value_counts().head(15).reset_index()
    top_stocks.columns = ["Stock", "Signal Count"]
    print(top_stocks.to_string(index=False))

    print("\n📂 Signals by Sector:")
    merged = results.merge(universe[["symbol", "sector"]], on="symbol", how="left")
    by_sector = merged.groupby("sector").size().sort_values(ascending=False).head(15)
    print(by_sector.to_string())

---

## Section 7 — 📈 Charts

Visual summaries of the backtest results. Charts are also saved to the `results/` folder.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

if results.empty:
    print("No results to chart.")
else:
    sns.set_theme(style="whitegrid", palette="muted")

    # ─── Chart 1: Return Distribution at 4 horizons ───────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Triple Confirmation Pattern — Forward Return Distributions",
                 fontsize=15, fontweight="bold", y=1.02)

    horizons = [
        ("ret_1m",  "1 Month Return",  "#2196F3"),
        ("ret_3m",  "3 Month Return",  "#4CAF50"),
        ("ret_6m",  "6 Month Return",  "#FF9800"),
        ("ret_1y",  "1 Year Return",   "#9C27B0"),
    ]

    for ax, (col, label, colour) in zip(axes.flat, horizons):
        clean = results[col].dropna()
        if len(clean) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(label)
            continue
        # Clip extreme outliers for readability (±150%)
        clipped = clean.clip(-150, 150)
        win_pct = (clean > 0).mean() * 100
        avg_ret = clean.mean()

        ax.hist(clipped, bins=30, color=colour, alpha=0.7, edgecolor="white")
        ax.axvline(0, color="black", linewidth=1.5, linestyle="--", label="Break-even")
        ax.axvline(avg_ret, color="red", linewidth=1.5, linestyle="-",
                   label=f"Avg {avg_ret:+.1f}%")
        ax.set_title(f"{label}  |  Win rate: {win_pct:.1f}%  |  n={len(clean)}",
                     fontsize=11)
        ax.set_xlabel("Return (%)")
        ax.set_ylabel("Number of signals")
        ax.legend(fontsize=9)

    plt.tight_layout()
    chart1_path = "results/chart1_return_distributions.png"
    plt.savefig(chart1_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {chart1_path}")

    # ─── Chart 2: Signals per Year ────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    results["year"] = pd.to_datetime(results["signal_date"]).dt.year
    by_year = results.groupby("year").size()
    by_year.plot(kind="bar", ax=ax, color="#1976D2", edgecolor="white")
    ax.set_title("Triple Confirmation Signals per Year", fontsize=13, fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Number of Signals")
    ax.tick_params(axis="x", rotation=0)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.3,
                int(bar.get_height()),
                ha="center", va="bottom", fontsize=10)
    plt.tight_layout()
    chart2_path = "results/chart2_signals_per_year.png"
    plt.savefig(chart2_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {chart2_path}")

    # ─── Chart 3: Average Return by Year ─────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    avg_by_year = results.groupby("year")[["ret_3m", "ret_6m", "ret_1y"]].mean()
    avg_by_year.columns = ["3M Avg Return", "6M Avg Return", "1Y Avg Return"]
    avg_by_year.plot(kind="bar", ax=ax, edgecolor="white")
    ax.axhline(0, color="black", linewidth=1, linestyle="--")
    ax.set_title("Average Forward Returns per Signal Year", fontsize=13, fontweight="bold")
    ax.set_xlabel("Signal Year")
    ax.set_ylabel("Average Return (%)")
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%+.0f%%"))
    ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="upper right")
    plt.tight_layout()
    chart3_path = "results/chart3_avg_returns_by_year.png"
    plt.savefig(chart3_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {chart3_path}")

    print("\n✅ All charts saved to the results/ folder.")
    print("   Right-click any file in the left panel and choose Download to save it.")

---

## ✅ Backtest Complete!

Your results are in the `results/` folder:

| File | Contents |
|---|---|
| `signals_YYYY_YYYY.csv` | All signals with forward returns (open in Excel) |
| `signals_YYYY_YYYY.xlsx` | Same data in Excel format |
| `chart1_return_distributions.png` | Histogram of returns at each horizon |
| `chart2_signals_per_year.png` | How many signals fired each year |
| `chart3_avg_returns_by_year.png` | Average returns grouped by year |
| `failed_downloads.txt` | Stocks that couldn't be downloaded (if any) |

### Column Guide for the CSV/Excel file

| Column | Meaning |
|---|---|
| `symbol` | NSE ticker (e.g. RELIANCE.NS) |
| `signal_date` | The week the Triple Confirmation fired |
| `signal_price` | Closing price on signal week (₹) |
| `stage` | Weinstein Stage at time of signal (should mostly be 2) |
| `mansfield_rs` | Relative Strength vs CNX500 (positive = outperforming) |
| `ret_1m` | % return 4 weeks after signal |
| `ret_3m` | % return 13 weeks after signal |
| `ret_6m` | % return 26 weeks after signal |
| `ret_1y` | % return 52 weeks after signal |

> **Important disclaimer:** This backtest is for research and education only. It does not account for brokerage costs, taxes, or liquidity. Past performance does not guarantee future results.

---

### Need to verify a signal against TradingView?

1. Open TradingView → search for the stock (remove `.NS`, e.g. search `RELIANCE`)
2. Set chart to **Weekly** timeframe
3. Add the Pine Script indicator
4. Navigate to the signal date from the CSV
5. Confirm the triangle marker (▲) appears on the same week